# 7.31 — Neural rendering & NeRF

NeRF renders a scene by asking a neural field what color and density live along each camera ray, then compositing those answers front to back.

This is a gap topic, so the notebook stays grounded in the lesson's ray equation $r(t)=o+td$ and transmittance compositing rather than training a full neural field. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(731)
np.set_printoptions(precision=4, suppress=True)

## Build volume rendering once

For samples along a ray, NeRF composites colors front to back:

$$C(r)=\sum_i T_i\left(1-e^{-\sigma_i\delta_i}ight)c_i,\qquad T_i=\prod_{j\lt i}e^{-\sigma_j\delta_j}$$

Opacity comes from density and interval length; transmittance says how much light survives to each sample.

In [ ]:
def ray_points(origin, direction, t_values):
    origin = np.asarray(origin, dtype=float)
    direction = np.asarray(direction, dtype=float)
    t_values = np.asarray(t_values, dtype=float)
    return origin[None, :] + t_values[:, None] * direction[None, :]


def render_ray(densities, colors, delta):
    densities = np.asarray(densities, dtype=float)
    colors = np.asarray(colors, dtype=float)
    alpha = 1.0 - np.exp(-densities * delta)
    transmittance = []
    running = 1.0
    for value in alpha:
        transmittance.append(running)
        running = running * (1.0 - value)
    transmittance = np.array(transmittance)
    weights = transmittance * alpha
    rgb = weights @ colors
    return alpha, transmittance, weights, rgb

points = ray_points([0, 0], [1, 0.5], [0, 1, 2, 3])
alpha_demo, trans_demo, weights_demo, rgb_demo = render_ray(
    [0.1, 1.0, 3.0],
    np.eye(3),
    0.5,
)
alpha_color, trans_color, weights_color, rgb_color = render_ray(
    [0.44628710262841953, 1.3862943611198906, 0.21072103131565256],
    np.eye(3),
    0.5,
)
print("ray point at t=2", points[2])
print("opacities", alpha_demo)
print("transmittance", trans_color)
print("weights", weights_color)
print("rgb", rgb_color)
assert np.allclose(points[2], [2, 1])
assert np.isclose(alpha_demo[0], 0.048770575499285984)
assert np.isclose(alpha_demo[2], 0.7768698398515702)
assert np.allclose(trans_color, [1.0, 0.8, 0.4])
assert np.allclose(weights_color, [0.2, 0.4, 0.04])
assert np.allclose(rgb_color, [0.2, 0.4, 0.04])

The exact alpha values $0.0488$ and $0.7769$ show why density is not a hard surface flag. It becomes visibility only through $1-e^{-\sigma\delta}$ and the surviving transmittance.

In [ ]:
def gaussian_field(points, center, color, density, width):
    diff = points - np.asarray(center, dtype=float)
    dist2 = (diff ** 2).sum(axis=1)
    sigma = density * np.exp(-dist2 / (2.0 * width ** 2))
    colors = np.repeat(np.asarray(color, dtype=float)[None, :], len(points), axis=0)
    return sigma, colors


def render_scene(origin, angle, objects, n_samples):
    direction = np.array([np.cos(angle), np.sin(angle)])
    t_values = np.linspace(0.0, 1.6, n_samples)
    points = ray_points(origin, direction, t_values)
    densities = np.zeros(n_samples)
    colors = np.zeros((n_samples, 3))
    for obj in objects:
        sigma, obj_colors = gaussian_field(points, obj["center"], obj["color"], obj["density"], obj["width"])
        colors = colors + sigma[:, None] * obj_colors
        densities = densities + sigma
    colors = colors / np.maximum(densities[:, None], 1e-9)
    delta = float(t_values[1] - t_values[0])
    alpha, trans, weights, rgb = render_ray(densities, colors, delta)
    return points, densities, alpha, weights, rgb


def make_scene_rung(name, objects, samples, angles):
    return {"name": name, "objects": objects, "samples": samples, "angles": angles}

rungs = [
    make_scene_rung("D1 one blob", [{"center": [0.8, 0.0], "color": [1, 0, 0], "density": 8, "width": 0.12}], 24, np.linspace(-0.25, 0.25, 21)),
    make_scene_rung("D2 two colors", [{"center": [0.75, -0.18], "color": [1, 0, 0], "density": 8, "width": 0.10}, {"center": [0.9, 0.22], "color": [0, 1, 0], "density": 7, "width": 0.10}], 28, np.linspace(-0.35, 0.35, 25)),
    make_scene_rung("D3 occluder", [{"center": [0.55, 0.0], "color": [0.05, 0.05, 0.05], "density": 9, "width": 0.08}, {"center": [1.05, 0.0], "color": [0, 0, 1], "density": 7, "width": 0.12}], 32, np.linspace(-0.25, 0.25, 25)),
    make_scene_rung("D4 thin target", [{"center": [0.8, -0.16], "color": [1, 0, 0], "density": 6, "width": 0.05}, {"center": [1.0, 0.18], "color": [0, 1, 0], "density": 6, "width": 0.05}, {"center": [1.2, 0.0], "color": [0, 0, 1], "density": 5, "width": 0.05}], 36, np.linspace(-0.35, 0.35, 31)),
    make_scene_rung("D5 cluttered rays", [{"center": [0.45, 0.0], "color": [0.2, 0.2, 0.2], "density": 10, "width": 0.07}, {"center": [0.85, -0.16], "color": [1, 0, 0], "density": 6, "width": 0.05}, {"center": [0.95, 0.18], "color": [0, 1, 0], "density": 6, "width": 0.05}, {"center": [1.25, 0.02], "color": [0, 0, 1], "density": 8, "width": 0.04}], 44, np.linspace(-0.38, 0.38, 35)),
]
for rung in rungs:
    print(f"{rung['name']:18s} objects={len(rung['objects'])} samples={rung['samples']} rays={len(rung['angles'])}")

In [ ]:
def label_for_angle(rung, angle):
    origin = np.array([0.0, 0.0])
    direction = np.array([np.cos(angle), np.sin(angle)])
    best_density = -1.0
    best_label = 0
    for obj in rung["objects"]:
        center = np.asarray(obj["center"], dtype=float)
        t = center @ direction
        if t <= 0.0:
            continue
        closest = t * direction
        distance = np.linalg.norm(center - closest)
        score = obj["density"] * np.exp(-(distance ** 2) / (2.0 * obj["width"] ** 2))
        if score > best_density:
            best_density = score
            best_label = int(np.argmax(obj["color"]))
    return best_label


def rendering_accuracy(rung):
    preds = []
    labels = []
    for angle in rung["angles"]:
        _, _, _, _, rgb = render_scene(np.array([0.0, 0.0]), angle, rung["objects"], rung["samples"])
        preds.append(int(np.argmax(rgb)))
        labels.append(label_for_angle(rung, angle))
    preds = np.array(preds)
    labels = np.array(labels)
    return float((preds == labels).mean())

metrics = []
for rung in rungs:
    acc = rendering_accuracy(rung)
    metrics.append(acc)
    print(f"{rung['name']:18s} rendering-hit accuracy={acc:.3f}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 5))
for idx, rung in enumerate(rungs):
    angle = rung["angles"][len(rung["angles"]) // 2]
    points, densities, alpha, weights, rgb = render_scene(np.array([0.0, 0.0]), angle, rung["objects"], rung["samples"])
    axes[0, idx].scatter(points[:, 0], points[:, 1], c=weights, cmap="viridis", s=20)
    for obj in rung["objects"]:
        axes[0, idx].scatter(obj["center"][0], obj["center"][1], c=[obj["color"]], s=80, marker="x")
    axes[0, idx].set_title(rung["name"].split()[0])
    axes[0, idx].set_xlim(0, 1.5)
    axes[0, idx].set_ylim(-0.45, 0.45)
    axes[0, idx].set_aspect("equal")
axes[1, 0].plot(range(1, 6), metrics, marker="o")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("accuracy")
axes[1, 0].set_title("Rendering-hit accuracy")
for idx in range(1, 5):
    axes[1, idx].axis("off")
fig.tight_layout()

## Pitfall on D5: forgetting transmittance

If we sum $lpha_i c_i$ without $T_i$, colors behind foreground density shine through too strongly. Front-to-back compositing discounts hidden samples by the light that survived previous samples.

In [ ]:
def render_without_transmittance(densities, colors, delta):
    alpha = 1.0 - np.exp(-densities * delta)
    rgb = alpha @ colors
    return rgb

rung = rungs[-1]
angle = 0.0
points, densities, alpha, weights, rgb = render_scene(np.array([0.0, 0.0]), angle, rung["objects"], rung["samples"])
colors = np.zeros((len(points), 3))
for obj in rung["objects"]:
    sigma, obj_colors = gaussian_field(points, obj["center"], obj["color"], obj["density"], obj["width"])
    colors = colors + sigma[:, None] * obj_colors
colors = colors / np.maximum(densities[:, None], 1e-9)
delta = 1.6 / (rung["samples"] - 1)
wrong_rgb = render_without_transmittance(densities, colors, delta)
print("wrong no-T rgb", np.round(wrong_rgb, 3))
print("fixed rgb", np.round(rgb, 3))
print("hidden blue excess", round(float(wrong_rgb[2] - rgb[2]), 3))
assert wrong_rgb[2] > rgb[2]

## Evaluate it + practice

- Track rendering-hit accuracy and compare with a no-skill majority color baseline.
- Sanity check: render a one-blob scene with many samples; the dominant color should match the blob.
- Ablation: remove transmittance; hidden samples become too visible.
- Failure signals: thin structures vanish, pose/ray errors blur colors, or interval length changes brightness.

Practice: halve the sample count in D5 and inspect which rays miss the thin blue object.

In [ ]:
# Your experiment here

Practice: move the foreground occluder and compare the fixed and no-transmittance RGB values.

In [ ]:
# Your experiment here

Practice: change $\delta$ and verify that opacity changes with interval length.

In [ ]:
# Your experiment here